In [ ]:
import iris
import matplotlib.pyplot as plt
import iris.plot as iplt 
import cartopy.crs as ccrs

<b>Specify the location of the files you wish to plot. </b><br>
UMfile -> Standard model data from the UM<br>
LFRic_slam_umified -> Post processed LFRic data that has been transformed to a structured grid using the slam utility. Meta data has been changed to mimick the UM meta data structures.<br>
LFRic_slam_only -> Post processed LFRic data that has been transformed to a structured grid using the slam utility. Meta data has not been changed from the original LFRic output. <br>

In [ ]:
path = '/PATH/TO/MY/RAL-LFRIC/OUTPUT'
#path = '/g/data/access/projects/access/lfric/workshop-2024/canned-output/share/cycle/20150603T0000Z'
UMfile = path+'/RMED-test/DAR/RAL3P2/um/umnsaa_pvera000'
LFRic_slam_umified = path+'/lfric_outputs/20150603T0000Z_lfric_slam_umf_s_006.nc'
LFRic_slam_only = path+'/lfric_outputs/20150603T0000Z_lfric_slam_s_006.nc'

The command to load the cubes in iris is the same for all three data formats. <br>
<i> Note that loading netcdf files is likely to throw a lot of warnings. You can ignore these or switch them off. </i>

In [ ]:
#switching off warnings related to the datum
import warnings
warnings.filterwarnings("ignore", message="Ignoring a datum in netCDF load for consistency with existing behaviour. In a future version of Iris, this datum will be applied. To apply the datum when loading, use the iris.FUTURE.datum_support flag.")

# UM data only created if suite run with 'UMIFY=true'
UMcubes = iris.load(UMfile)

In [ ]:
LFRic_slam_UM = iris.load(LFRic_slam_umified)
LFRic_slam = iris.load(LFRic_slam_only)

Print the cube lists for LFRic_slam and LFRic_slam_umified. Note the different coordinate names. <br>
You can also see differences on the cube level, e.g., the addition of a forecast_reference_time and a STASH code, which you'd expect to find in a UM data file. <br> 

In [ ]:
print(LFRic_slam[:5], '\n')

In [ ]:
print(LFRic_slam_UM[:5])

In [ ]:
print(LFRic_slam.extract_cube('temperature_at_screen_level'), '\n')

In [ ]:
print(LFRic_slam_UM.extract_cube('temperature_at_screen_level'), '\n')
print(UMcubes.extract_cube('air_temperature'))

<b> Plotting the data </b><br>
Thanks to the structured grid, there is no difference in plotting the three data formats in matplotlib. During the slam conversion, the rotated pole coordinates are written to the file's meta data, so that coastlines can be added the same way you would add them for the UM. 

In [ ]:
def plot_field(cube):
    ''' Takes a 2d iris cube and plots it '''
    if type(cube.coord_system()) == iris.coord_systems.RotatedGeogCS:
        plat = cube.coord_system().grid_north_pole_latitude
        plon = cube.coord_system().grid_north_pole_longitude
    else:
        plat = 90.0
        plon = 180.0
    prjn = ccrs.RotatedPole(pole_longitude=plon, pole_latitude=plat)

    fig = plt.figure()
    ax = fig.add_subplot(1, 1, 1, projection=prjn)
    
    iplt.pcolormesh(cube)
    plt.colorbar()

    ax.coastlines('10m')
    
    plt.show()

In [ ]:
plot_field(UMcubes.extract_cube('air_temperature')[2])

In [ ]:
plot_field(LFRic_slam_UM.extract_cube('temperature_at_screen_level')[1])

In [ ]:
plot_field(LFRic_slam.extract_cube('temperature_at_screen_level')[1])

<b>UM - LFRic differences </b><br>
Iris does not support mathematical operations (such as differences) between cubes where the meta data differs. However, differences between UM and LFRic can be calculated using <br>
cube1 - cube2.data<br>
Important: When doing this, please implement the necessary checks to ensure you are indeed comparing like with like (such as matching times).
<p>
Note that UM and LFRic data are "upside down" from each other. This seems to be due to a difference in how data arrays are loaded from netcdf vs pp/fields files. One of the arrays has to be "flipped over" to get a meaningful difference plot. 

In [ ]:
UM_temp = UMcubes.extract_cube('air_temperature')[1]
LFRic_temp = LFRic_slam_UM.extract_cube('temperature_at_screen_level')[0]
checking that the times match:
print(UM_temp.coord('time'))
print(LFRic_temp.coord('time'))

In [ ]:
#calculating the difference without flipping the array
difference = LFRic_temp - UM_temp.data
data_abs_max = max(abs(difference.data.max()),abs(difference.data.min()))
iplt.pcolormesh(difference, cmap='seismic', vmin=-data_abs_max, vmax=data_abs_max)
plt.colorbar()
plt.show()

In [ ]:
#flipping the UM_temp data array:
difference = LFRic_temp - UM_temp.data[::-1]
data_abs_max = max(abs(difference.data.max()),abs(difference.data.min()))
iplt.pcolormesh(difference, cmap='seismic', vmin=-data_abs_max, vmax=data_abs_max)
plt.colorbar()
plt.show()